In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import ssl
import urllib.request
ssl._create_default_https_context = ssl._create_unverified_context

# 1. Set up Hyperparameters and Device
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 15
# Automatically use GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Download and Load the MNIST Dataset
# Transforms convert the raw images to PyTorch Tensors and normalize the pixel values (0 to 1)
transform = transforms.Compose([
    transforms.RandomRotation(degrees=15),          # Randomly tilt the image up to 15 degrees
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)), # Shift and zoom
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 3. Define the Convolutional Neural Network (CNN)
class MNIST_CNN(nn.Module):
    def __init__(self):
        super(MNIST_CNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16) # Added Batch Normalization
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Conv Block 2
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32) # Added Batch Normalization
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Fully Connected Block
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(0.5) # Added Dropout (ignores 50% of neurons during training)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = x.view(-1, 32 * 7 * 7)
        x = self.dropout(self.relu3(self.fc1(x)))
        x = self.fc2(x)
        return x
    
model = MNIST_CNN().to(device)

# 4. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 5. Training Loop
print("\nStarting Training...")
for epoch in range(EPOCHS):
    model.train() # Set model to training mode
    running_loss = 0.0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Zero out the gradients from the previous step
        optimizer.zero_grad()
        
        # Forward pass: make predictions
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass: calculate gradients
        loss.backward()
        
        # Update model weights
        optimizer.step()
        
        running_loss += loss.item()
        
        if (i+1) % 300 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

# 6. Evaluation (Testing) Loop
print("\nStarting Testing...")
model.eval() # Set model to evaluation mode (disables dropout/batchnorm updates)
correct = 0
total = 0

# Disable gradient calculation for testing to save memory and speed up computation
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        
        # Get the index of the highest probability class
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy on the 10,000 test images: {accuracy:.2f}%")

Training on device: cpu

Starting Training...
Epoch [1/15], Step [300/938], Loss: 0.4669
Epoch [1/15], Step [600/938], Loss: 0.4539
Epoch [1/15], Step [900/938], Loss: 0.4433
Epoch [2/15], Step [300/938], Loss: 0.1901
Epoch [2/15], Step [600/938], Loss: 0.5160
Epoch [2/15], Step [900/938], Loss: 0.1289
Epoch [3/15], Step [300/938], Loss: 0.2822
Epoch [3/15], Step [600/938], Loss: 0.2180
Epoch [3/15], Step [900/938], Loss: 0.0851
Epoch [4/15], Step [300/938], Loss: 0.1481
Epoch [4/15], Step [600/938], Loss: 0.1144
Epoch [4/15], Step [900/938], Loss: 0.2625
Epoch [5/15], Step [300/938], Loss: 0.1179
Epoch [5/15], Step [600/938], Loss: 0.2565
Epoch [5/15], Step [900/938], Loss: 0.1761
Epoch [6/15], Step [300/938], Loss: 0.0956
Epoch [6/15], Step [600/938], Loss: 0.0985
Epoch [6/15], Step [900/938], Loss: 0.1051
Epoch [7/15], Step [300/938], Loss: 0.0732
Epoch [7/15], Step [600/938], Loss: 0.3222
Epoch [7/15], Step [900/938], Loss: 0.2769
Epoch [8/15], Step [300/938], Loss: 0.2403
Epoch [8

In [6]:
torch.save(model.state_dict(), 'mnist_cnn.pth')
print("Model saved to mnist_cnn.pth")

Model saved to mnist_cnn.pth
